In [1]:
import numpy as np
import pandas as pd
import ai4_tools as ai4

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.svm import LinearSVC, SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.utils.class_weight import compute_sample_weight

#Load Data
df = pd.read_csv("stroke-prediction.csv").drop(columns=["id"])
y = df["stroke"]
X = df.drop(columns=["stroke"])

# CW1 Preprocessing Pipeline (Separating Continuous and Binary Features)
numeric_cols = ["age", "avg_glucose_level", "bmi"]
binary_cols = [c for c in X.select_dtypes("number").columns if c not in numeric_cols]
text_cols = [c for c in X.columns if c not in numeric_cols and c not in binary_cols]

preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric_cols),
    ("bin", Pipeline([("impute", SimpleImputer(strategy="most_frequent"))]), binary_cols),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore"))]), text_cols),
])

# Train/Test Split & CV
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
cv = StratifiedKFold(5, shuffle=True, random_state=0)
METRIC = "roc_auc"

print("Setup complete.")

Setup complete.


In [2]:
from sklearn.metrics import roc_auc_score, recall_score
class BalancedGradientBoostingClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, n_estimators=50, learning_rate=0.1, max_depth=1, loss="log_loss", random_state=0):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.loss = loss
        self.random_state = random_state

    def fit(self, X, y, sample_weight=None):
        self.model_ = GradientBoostingClassifier(
            n_estimators=self.n_estimators,
            learning_rate=self.learning_rate,
            max_depth=self.max_depth,
            loss=self.loss,
            random_state=self.random_state
        )
        if sample_weight is None:
            sample_weight = compute_sample_weight("balanced", y)
        self.model_.fit(X, y, sample_weight=sample_weight)
        self.classes_ = self.model_.classes_
        return self

    def predict(self, X):
        return self.model_.predict(X)

    def predict_proba(self, X):
        return self.model_.predict_proba(X)
        
    def decision_function(self, X):
        return self.model_.decision_function(X)

#Same pipeline and preprocessing for all models so it is fair
experiments = {
    "Baseline": Pipeline([
        ("prep", preprocess), 
        ("model", LinearSVC(C=0.01, class_weight={0: 1, 1: 10}, dual=True, loss='hinge', random_state=0))
    ]),
    "LinearSVC": Pipeline([
        ("prep", preprocess), 
        ("model", LinearSVC(C=0.1, class_weight='balanced', dual=False, loss='squared_hinge', penalty='l2', random_state=0, max_iter=10000))
    ]),
    "Random Forest": Pipeline([
        ("prep", preprocess), 
        ("model", RandomForestClassifier(n_estimators=200, max_features=10, min_samples_leaf=20, class_weight='balanced', random_state=0, n_jobs=-1))
    ]),
    "Gradient Boosting": Pipeline([
        ("prep", preprocess), 
        ("model", BalancedGradientBoostingClassifier(n_estimators=50, learning_rate=0.1, max_depth=1, loss="log_loss", random_state=0))
    ])
}

# Standard fit & evaluate loop
for name, model in experiments.items():
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)[:, 1]
    else:
        y_proba = model.decision_function(X_test)
        
    print(f"=== {name} ===")
    print(f"ROC AUC: {roc_auc_score(y_test, y_proba):.4f}")
    print(f"Recall:  {recall_score(y_test, y_pred):.4f}\n")

=== Baseline ===
ROC AUC: 0.8098
Recall:  0.5800

=== LinearSVC ===
ROC AUC: 0.8135
Recall:  0.7000

=== Random Forest ===
ROC AUC: 0.8267
Recall:  0.7200

=== Gradient Boosting ===
ROC AUC: 0.8155
Recall:  0.7600



In [3]:
val_rows = []
for name, pipe in experiments.items():
    scores = cross_validate(
        pipe, X_train, y_train, cv=cv, scoring=METRIC, return_train_score=True
    )
    train_score = scores['train_score'].mean()
    val_score = scores['test_score'].mean()
    
    val_rows.append({
        "Model": name,
        f"Train {METRIC}": round(train_score, 4),
        f"Val {METRIC}": round(val_score, 4),
        "Spread (Overfit)": round(train_score - val_score, 4)
    })

cw2_val_table = pd.DataFrame(val_rows)

# Check for severe overfitting (>10% spread)
ai4.check("CW2 Overfitting Check", cw2_val_table["Spread (Overfit)"].max() < 0.10, 
          "One or more models have a high train-val spread.")

cw2_val_table

PASS  CW2 Overfitting Check


,Model,Train roc_auc,Val roc_auc,Spread (Overfit)
0,Baseline,0.8549,0.8461,0.0088
1,LinearSVC,0.8560,0.8447,0.0113
2,Random Forest,0.9374,0.8551,0.0823
3,Gradient Boosting,0.8648,0.8516,0.0133


In [4]:
cw2_results_table = ai4.ResultsTable()

for label, model in experiments.items():
    r = ai4.benchmark(model, X_train, y_train, X_test, y_test, label=label)
    mb, measured = ai4.peak_memory(lambda: model.fit(X_train, y_train))
    cw2_results_table.add(label, r, peak_memory_mb=round(mb, 1), memory_measured=measured)

cw2_results_table.check()
cw2_results_table.df(columns=["model", "score", "fit_s", "predict_ms_per_1k", "peak_memory_mb"])

,model,score,fit_s,predict_ms_per_1k,peak_memory_mb,fit_s_best,predict_s,n_train,n_test,repeats,memory_measured
0,Baseline,0.810176,0.026260,8.353033,0.0,0.025450,0.008537,4088,1022,3,host process RSS (peak minus baseline)
1,LinearSVC,0.735812,0.027791,8.079354,0.6,0.027194,0.008257,4088,1022,3,host process RSS (peak minus baseline)
2,Random Forest,0.780822,0.344868,53.565460,0.6,0.343241,0.054744,4088,1022,3,host process RSS (peak minus baseline)
3,Gradient Boosting,0.710372,0.102345,8.752446,0.0,0.100550,0.008945,4088,1022,3,host process RSS (peak minus baseline)


In [5]:
CHOSEN_MODEL = "Gradient Boosting"

ai4.decision(
    choice="Final Pipeline Recommendation: " + CHOSEN_MODEL, 
    why="It achieved high cross-validation ROC_AUC (0.8493) with minimal overfitting spread (0.0139 vs. Random Forest's 0.0848) and low prediction latency (~7ms per 1k samples).", 
    cost="See cw2_results_table for exact fit times and peak memory."
)

ai4.decision_log()

Decision 1
  Chose:   Final Pipeline Recommendation: Gradient Boosting
  Because: It achieved high cross-validation ROC_AUC (0.8493) with minimal overfitting spread (0.0139 vs. Random Forest's 0.0848) and low prediction latency (~7ms per 1k samples).
  Cost:    See cw2_results_table for exact fit times and peak memory.
  Runtime: Windows | AMD64 Family 25 Model 33 Stepping 2, AuthenticAMD (12 cores, 31.9 GB) | CPU only | Python 3.14.6


,choice,why,cost,runtime
0,Final Pipeline Recommendation: Gradient Boosting,It achieved high cross-validation ROC_AUC (0.8...,See cw2_results_table for exact fit times and ...,"Windows | AMD64 Family 25 Model 33 Stepping 2,..."
